In [0]:
%run ./00_import


In [0]:
%run ./01_env


In [0]:
%run ./02_load_data

In [0]:
fact_monitoring_t = (
    fact_monitoring
    .withColumn("reco_date", F.to_date("date_creation_reco"))
    .withColumn(
        "period_flag",
        F.when(
            (F.col("reco_date") >= F.date_sub(F.current_date(), 30)) &
            (F.col("reco_date") < F.current_date()),
            "CURRENT_30_DAYS"
        )
        .when(
            (F.col("reco_date") >= F.date_sub(F.current_date(), 60)) &
            (F.col("reco_date") < F.date_sub(F.current_date(), 30)),
            "PREVIOUS_30_DAYS"
        )
        .otherwise("OUT_OF_SCOPE")
    )
)

In [0]:
monitoring_current = fact_monitoring_t.filter(F.col("period_flag") == "CURRENT_30_DAYS")
monitoring_previous = fact_monitoring_t.filter(F.col("period_flag") == "PREVIOUS_30_DAYS")

In [0]:
#Avoir que les status non deleted
df_batch_status_t = (
    df_batch_status
    .filter(F.col("deleted") == False)
    .select(
        F.col("id_parameter_batch_status").alias("batch_status_id"),
        F.col("code").alias("batch_status_code")
    )
)

df_batch_completed = (
    df_batches
    .filter(F.col("deleted") == False)

    # join status
    .join(
        df_batch_status_t,
        df_batches["status"] == df_batch_status_t["batch_status_id"],
        "inner"
    )

    # join production lines
    .join(
        df_pl.select(
            F.col("id_plant_production_line").alias("production_line_id"),
            F.col("common_code").alias("production_line_code")
        ),
        F.col("production_line") == F.col("production_line_id"),
        "left"
    )

    # uniquement batchs terminés
    .filter(F.col("batch_status_code") == "finished")

    .select(
        F.col("id_batch").alias("batch_id"),
        "fabrication_order",
        "production_line_code",
        "batch_status_code"
    )
)

In [0]:
#Données de qualité obligatoire pour l'IA
mandatory_features = [
    "malt_soluble_beta_glucan",
    "malt_color_ebc",
    "malt_friability",
    "malt_free_amino_nitrogen",
    "malt_moisture"
]

#Union Données de qualité obligatoires pour l'IA des différentes lignes de production 
df_features_all = (
    df_features_ro1.withColumn("production_line_code", F.lit("RO1"))
    .unionByName(df_features_ng1.withColumn("production_line_code", F.lit("NG1")))
    .unionByName(df_features_ng2.withColumn("production_line_code", F.lit("NG2")))
    .unionByName(df_features_pr1.withColumn("production_line_code", F.lit("PR1")))
    .unionByName(df_features_st1.withColumn("production_line_code", F.lit("ST1")))
    .unionByName(df_features_st2.withColumn("production_line_code", F.lit("ST2")))
    .unionByName(df_features_po1.withColumn("production_line_code", F.lit("PO1")))
    .filter(F.col("feature_reference").isin(mandatory_features))
    .select("production_line_code", "feature_reference")
    .dropDuplicates()
)





#Union Données de qualité de malt
df_quality_results_all = (
    df_quality_results_ro1.withColumn("production_line_code", F.lit("RO1"))
    .unionByName(df_quality_results_ng1.withColumn("production_line_code", F.lit("NG1")))
    .unionByName(df_quality_results_ng2.withColumn("production_line_code", F.lit("NG2")))
    .unionByName(df_quality_results_pr1.withColumn("production_line_code", F.lit("PR1")))
    .unionByName(df_quality_results_st1.withColumn("production_line_code", F.lit("ST1")))
    .unionByName(df_quality_results_st2.withColumn("production_line_code", F.lit("ST2")))
    .unionByName(df_quality_results_po1.withColumn("production_line_code", F.lit("PO1")))
)

#Transformer les colonnes de qualité (dans result) en lignes 
df_quality_results_long = df_quality_results_all.selectExpr(
    "fabrication_order",
    "production_line_code",
    """
    stack(
        5,
        'malt_soluble_beta_glucan', malt_soluble_beta_glucan,
        'malt_color_ebc', malt_color_ebc,
        'malt_friability', malt_friability,
        'malt_free_amino_nitrogen', malt_free_amino_nitrogen,
        'malt_moisture', malt_moisture
    ) as (feature_reference, quality_value)
    """
)







